## Requisitos del entorno

Ejecutar este notebook en un entorno con `pandas`, `numpy`, `scikit-learn`, `statsmodels`, `seaborn`, `matplotlib`, `plotly`, `openpyxl`, `joblib` y `gdown` previamente instalados.

> **Secuencia de ejecución CRISP-DM:**
> 1. `01_carga_datos.ipynb — Fase 1: Comprensión del negocio + Carga de datos`
> 2. `02_limpieza_variables.ipynb — Fase 2: Comprensión y preparación de datos`
> 3. `03_universo_analitico.ipynb — Fase 3: Preparación de datos (integración y variables económicas)`
> 4. `04_modelado.ipynb — Fase 4: Modelado (WLS + KMeans)`
> 5. `05_modelo_hibrido.ipynb — Fase 4b: Modelo Híbrido (residuos + clasificador)`
> 6. `06_evaluacion_implementacion.ipynb — Fase 5-6: Evaluación + Implementación`

> ➡️ **Siguiente notebook: `02_limpieza_variables.ipynb`**

![Universidad Central](https://universidad.ucentral.edu.co/tulengua/wp-content/themes/tulengua/images/logo-ucentral.png)

<h2 align="center">Procesamiento y análisis de datos</h2>

<table>
<tr>
<td style="width: 75%; vertical-align: middle;">

## Estimación del gasto personal como variable principal para la evaluación de la capacidad de endeudamiento a partir de la caracterización de los hogares desde la analítica de datos.

**FASE 1 — Comprensión del Negocio + Carga de Microdatos**

**CRISP-DM Fase 1:** Comprensión del negocio y conexión/carga de microdatos desde fuentes externas (ENPH 2016-2017).

</td>
<td style="width: 25%; text-align: center;">

<img src="https://raw.githubusercontent.com/lpaolav/bases-de-datos/main/gasto_personal.png" width="150">

</td>
</tr>
</table>

---

### Estudiantes:
- Oscar Leonardo Duarte Urrego  
- Paola Andrea Velandia Lozano  

### Director de Tesis:
- Miguel Hernández Bejarano

---


In [1]:

# ============================================================
# CONFIGURACIÓN DE PERSISTENCIA
# ============================================================
from pathlib import Path
import os

# Directorio de datos intermedios (relativo al proyecto)
_cwd = Path.cwd()
BASE_PATH = Path(os.getenv("TESIS_BASE_PATH",
    _cwd.parent if _cwd.name in ("notebooks", "notebook") else _cwd)).resolve()
PERSIST_DIR = BASE_PATH / "02_intermedios"
PERSIST_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Directorio de persistencia: {PERSIST_DIR}")


📁 Directorio de persistencia: /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/02_intermedios


## Requisitos de entorno

Ejecutar este notebook en un entorno con `pandas`, `numpy`, `scikit-learn`, `statsmodels`, `seaborn`, `matplotlib`, `plotly`, `openpyxl`, `joblib` y `gdown` previamente instalados. Las instalaciones se dejan fuera del notebook para evitar dependencias ocultas y mejorar la reproducibilidad.


In [2]:
import sys
print(sys.executable)

/Users/paolavelandia/anaconda3/bin/python


In [3]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib

![Universidad Central](https://universidad.ucentral.edu.co/tulengua/wp-content/themes/tulengua/images/logo-ucentral.png)


<h2 align="center">Procesamiento y análisis de datos
</h2>

<table>
<tr>
<td style="width: 75%; vertical-align: middle;">

## Estimación del gasto personal como variable principal para la evaluación de la capacidad de endeudamiento a partir de la caracterización de los hogares desde la analítica de datos.

</td>
<td style="width: 25%; text-align: center;">

<img src="https://raw.githubusercontent.com/lpaolav/bases-de-datos/main/gasto_personal.png" width="150">

</td>
</tr>
</table>

---

### Estudiantes:
- Oscar Leonardo Duarte Urrego  
- Paola Andrea Velandia Lozano  

### Director de Tesis:
- Miguel Hernández Bejarano


---


El presente scrip, consolida de manera estructurada y reproducible el proceso de tratamiento de datos, análisis y modelado de los datos obtenidos de la Encuesta Naciona de Presupuesto de los Hogares Colombianos cifras correspondientes al segundo semestre de 2016 y primer semestre de 2017. La metodología orientada al desarrollo del proyecto es CRISP-DM (Cross Industry Standar Process for Data Mining), desarrollada a través de las siguientes fases: comprensión del negocio, comprensión de los datos, entendimiento y preparación de los datos, modelado, evaluación e implementación.

El scrip permite conocer la trazabilidad metodológica, reproducibilidad de los resultados y consistencia en los procesos analíticos para su posterior implementación. 

A continuación las fases del proyecto:

1. Conexión y carga de microdatos desde fuentes externas.
2. Proceso de limpieza, homologacion y construccion de variables analíticas.
3. Definición del universo de análisis y generacipon de variables económicas relevantes.
4. Estimación del modelo de regresión lineal multiple para la estimación del gasto personal.
5. Segmentación mediante técnicas de clusterización.
6. Construcción de artefactor de producción (modelos, transformaciones y parámetros) exportados en formato .pkl.
7. Simulación del gasto personal en un entorno de implementación.


 Guía de Reproducibilidad y Lectura

Este cuaderno fue organizado para tesis con cuatro objetivos:

1. Permitir ejecución secuencial de principio a fin.
2. Centralizar rutas, semillas y parámetros críticos.
3. Separar carga, depuración, construcción analítica, modelado y segmentación.
4. Dejar exportación de artefactos y simulación como etapas explícitas de implementación.



# **1. Conexión y carga de microdatos desde fuentes externas.**


En esta etapa se realiza la ingesta de los microdatos provenientes de fuentes externas, garantizando la trazabilidad y reproducibilidad del proceso. Los datos son descargados desde un repositorio en la nube (Google Drive) en formato comprimido, descomprimidos automáticamente y posteriormente cargados de manera recursiva en estructuras tipo DataFrame.

El procedimiento contempla la lectura de múltiples formatos de archivo (CSV, TXT y XLSX), aplicando configuraciones específicas de codificación y delimitadores según la naturaleza de cada fuente. Adicionalmente, se incorporan validaciones para evitar errores de lectura, normalización de nombres de variables y optimización de memoria mediante la conversión de variables categóricas. Por último, los datos son almacenados en formato eficiente (parquet), facilitando su reutilización en etapas posteriores sin reprocesar la información original.

Con el fin de dar cumplimiento a los criterios de evaluación y garantizar la transparencia en el uso de la información, se relaciona el acceso a la fuente oficial de los microdatos utilizados en el estudio:

Departamento Administrativo Nacional de Estadística (DANE). (2024). Encuesta Nacional de Presupuestos de los Hogares (ENPH) Julio 2016–Julio 2017 [Conjunto de datos].
https://microdatos.dane.gov.co/index.php/catalog/566/get-microdata


In [4]:
# ======================================
# 0. CONFIGURACIÓN GENERAL Y LIBRERÍAS
# ======================================
from pathlib import Path
import os
import warnings
import zipfile

import numpy as np
import pandas as pd
from pandas.errors import SettingWithCopyWarning

try:
    import gdown
except ImportError as exc:
    raise ImportError(
        "Instala 'gdown' en el entorno antes de ejecutar este notebook."
    ) from exc

warnings.filterwarnings("ignore", category=SettingWithCopyWarning)
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message="The default of observed=False is deprecated*",
)
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message="Passing `palette` without assigning `hue` is deprecated*",
)
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message="The behavior of Series.replace*CategoricalDtype*",
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

SEED = int(os.getenv("TESIS_RANDOM_STATE", "42"))
CLUSTER_SEED = int(os.getenv("TESIS_CLUSTER_RANDOM_STATE", "90"))
N_CLUSTERS = int(os.getenv("TESIS_N_CLUSTERS", "4"))
TARGET_PERIOD = int(os.getenv("TESIS_TARGET_PERIOD", "202512"))
FILE_ID = os.getenv("TESIS_GDRIVE_FILE_ID", "1UbjjqTIdf8gschIK0dbo2ZJCs6YSyH6_")

_cwd = Path.cwd()
BASE_PATH = Path(os.getenv("TESIS_BASE_PATH",
    _cwd.parent if _cwd.name in ("notebooks", "notebook") else _cwd)).resolve()
RAW_DIR = Path(os.getenv("TESIS_RAW_DIR", BASE_PATH / "01_datos" / "raw"))
OUTPUT_DIR = Path(os.getenv("TESIS_OUTPUT_DIR", BASE_PATH / "produccion"))
ARTIFACTS_DIR = Path(os.getenv("TESIS_ARTIFACT_DIR", OUTPUT_DIR / "modelos"))

RAW_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

zip_path = RAW_DIR / "datos_enph.zip"
extract_path = RAW_DIR / "dataset"


def normalizar_nombre(nombre):
    base = os.path.splitext(nombre)[0]
    return base.replace(" ", "_").lower()


def descargar_y_extraer_datos(file_id, zip_path, extract_path):
    url = f"https://drive.google.com/uc?id={file_id}"

    if not zip_path.exists():
        print("Descargando microdatos desde Google Drive...")
        gdown.download(url, str(zip_path), quiet=False)
    else:
        print(f"ZIP existente: {zip_path}")

    if not extract_path.exists():
        print("Descomprimiendo fuente de datos...")
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(extract_path)
    else:
        print(f"Directorio de extracción existente: {extract_path}")


def cargar_dataframes(extract_path):
    dataframes = {}
    errores = []

    print("\nCargando archivos fuente...\n")

    for root, _, files in os.walk(extract_path):
        for archivo in files:
            if archivo.startswith("._"):
                continue

            path = os.path.join(root, archivo)
            nombre_df = normalizar_nombre(archivo)

            if nombre_df in dataframes:
                nombre_df = f"{nombre_df}_{len(dataframes)}"

            try:
                if archivo.lower().endswith(".csv"):
                    df = pd.read_csv(
                        path,
                        sep=";",
                        encoding="utf-8",
                        on_bad_lines="skip",
                        low_memory=False,
                    )
                elif archivo.lower().endswith(".txt"):
                    try:
                        df = pd.read_csv(
                            path,
                            sep="|",
                            encoding="latin1",
                            on_bad_lines="skip",
                        )
                        if df.shape[1] == 1:
                            df = pd.read_csv(
                                path,
                                sep="\t",
                                encoding="latin1",
                                on_bad_lines="skip",
                            )
                    except Exception:
                        df = pd.read_csv(
                            path,
                            sep="\t",
                            encoding="latin1",
                            on_bad_lines="skip",
                        )
                elif archivo.lower().endswith(".xlsx"):
                    nombre_archivo = archivo.lower()

                    if "cargue_inflacion" in nombre_archivo:
                        df = pd.read_excel(
                            path,
                            sheet_name="IPC_BASE_2018",
                            dtype={
                                "Peso_IPC_base_2018": float,
                                "Var_Mes_base_2018": float,
                                "IPC_BASE_DIC_2018": float,
                            },
                        )
                    else:
                        df = pd.read_excel(path)
                else:
                    continue

                for col in df.select_dtypes(include=["object"]).columns:
                    df[col] = df[col].astype("category")

                dataframes[nombre_df] = df
            except Exception as exc:
                errores.append((archivo, str(exc)))

    print(f"DataFrames cargados: {len(dataframes)}")
    if errores:
        print(f"Archivos con error: {len(errores)}")
        for archivo, error in errores[:10]:
            print(f"- {archivo}: {error}")

    return dataframes, errores


descargar_y_extraer_datos(FILE_ID, zip_path, extract_path)
dataframes, errores = cargar_dataframes(extract_path)
list(dataframes.keys())

list(dataframes.keys())

CaracteristicasGP = dataframes['caracteristicas_generales_personas']
ViviendaHog = dataframes['viviendas_y_hogares']
GdUrbanoI = dataframes['gastos_diarios_urbanos']
GdPersonales = dataframes['gastos_diarios_personales_urbano']
GPersonComidasfueraH = dataframes['gastos_personales_urbano_-_comidas_preparadas_fuera_del_hogar']
GmfMP = dataframes['gastos_menos_frecuentes_-_medio_de_pago']
GmenosFreqUrbano = dataframes['gastos_menos_frecuentes_-_articulos']
GdComidasfueraH = dataframes['gastos_diarios_del_hogar_urbano_-_comidas_preparadas_fuera_del_hogar']
GdUrbanoC = dataframes['gastos_diarios_urbano_-_capitulo_c']
GdMercados = dataframes['gastos_diarios_urbanos_-_mercados']
homologa_producto = dataframes['homologa_producto']
IPC_BASE_DIC_2018 = dataframes["cargue_inflacion"]

ZIP existente: /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/01_datos/raw/datos_enph.zip
Directorio de extracción existente: /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/01_datos/raw/dataset

Cargando archivos fuente...



DataFrames cargados: 14


---
## 💾 Persistencia — Exportación NB01
Guarda todos los DataFrames cargados en formato `.parquet` para ser reutilizados por `02_limpieza_variables.ipynb`.

In [5]:

# ============================================================
# EXPORTACIÓN DE DATOS — FIN NB01
# Guarda los DataFrames fuente en parquet para NB02
# ============================================================
import joblib

# DataFrames grandes: parquet
for nombre, df in dataframes.items():
    ruta = PERSIST_DIR / f"raw_{nombre}.parquet"
    try:
        df_save = df.copy()
        for col in df_save.select_dtypes(include='category').columns:
            df_save[col] = df_save[col].astype(str)
        df_save.to_parquet(ruta, index=False)
    except Exception as e:
        print(f"  ⚠️  {nombre}: {e}")

print(f"✅ NB01 → {len(dataframes)} DataFrames guardados en {PERSIST_DIR}")


✅ NB01 → 14 DataFrames guardados en /Users/paolavelandia/01_a_T_E_S_I_S/01_tesis_v4/02_intermedios
